# LeetCode 1002 — Find Common Characters bằng Python

**Link đề:** https://leetcode.com/problems/find-common-characters/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, String

```python
class Solution:
    def commonChars(self, words: list[str]) -> list[str]:
        ...
```

## Mục tiêu học
1. Hiểu "ký tự chung **kể cả lặp lại**" = lấy **số lần xuất hiện nhỏ nhất** của mỗi chữ cái qua mọi từ.
2. Thành thạo **đếm tần suất** bằng mảng 26 ô (`ord(ch) - ord('a')`) và bằng `collections.Counter`.
3. Biết phép **giao multiset** `Counter & Counter` và `functools.reduce`.
4. Viết 4 cách: brute force xoá dần, mảng 26 ô lấy min, Counter `&`, đếm theo `str.count`.
5. Trace, chứng minh, phân tích Time/Space, và làm 6 bài tập tăng dần.

## 1. Đề bài và ví dụ

Cho mảng chuỗi `words`, trả về list **mọi ký tự xuất hiện trong tất cả** các chuỗi, **tính cả số lần lặp**. Thứ tự kết quả tuỳ ý.

| words | Kết quả | Giải thích |
|---|---|---|
| `["bella","label","roller"]` | `["e","l","l"]` | `e` có trong cả 3 (ít nhất 1 lần); `l` có ít nhất **2** lần trong mỗi từ |
| `["cool","lock","cook"]` | `["c","o"]` | `o` có 2 lần trong "cool", "cook" nhưng chỉ 1 lần trong "lock" → lấy 1 |

**Constraints:** `1 <= len(words) <= 100`, `1 <= len(words[i]) <= 100`, chỉ chữ thường.

"Tính cả số lần lặp" là điểm mấu chốt: nếu mỗi từ đều có ít nhất `k` chữ `l`, ta phải trả về `k` chữ `l`.

In [1]:
words = ["bella", "label", "roller"]
for w in words:
    print(f"{w:7} -> số chữ 'l' = {w.count('l')}, số chữ 'e' = {w.count('e')}")
print("=> lấy min: 'l' x", min(w.count('l') for w in words), "| 'e' x", min(w.count('e') for w in words))

bella   -> số chữ 'l' = 2, số chữ 'e' = 1
label   -> số chữ 'l' = 2, số chữ 'e' = 1
roller  -> số chữ 'l' = 2, số chữ 'e' = 1
=> lấy min: 'l' x 2 | 'e' x 1


## 2. Nền tảng Python

### 2.1. `ord`, `chr` — đổi chữ cái thành chỉ số 0..25

- `ord(ch)` trả mã số của ký tự (`ord('a') = 97`).
- `ord(ch) - ord('a')` biến `'a'..'z'` thành `0..25` → dùng làm chỉ số mảng.
- `chr(i + ord('a'))` làm chiều ngược lại.

In [2]:
for ch in "abz":
    print(ch, ord(ch), "-> chỉ số", ord(ch) - ord('a'))
print("chỉ số 4 ->", chr(4 + ord('a')))

a 97 -> chỉ số 0
b 98 -> chỉ số 1
z 122 -> chỉ số 25
chỉ số 4 -> e


### 2.2. Mảng đếm 26 ô

In [3]:
def count26(word: str) -> list[int]:
    cnt = [0] * 26
    for ch in word:
        cnt[ord(ch) - ord('a')] += 1
    return cnt

c = count26("bella")
print(c)
print({chr(i + 97): v for i, v in enumerate(c) if v})

[1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
{'a': 1, 'b': 1, 'e': 1, 'l': 2}


### 2.3. `Counter` và phép giao `&`

`Counter("bella")` → `{'l': 2, 'b': 1, 'e': 1, 'a': 1}`.

Phép `c1 & c2` giữa hai Counter cho **số lần nhỏ nhất** của mỗi khóa (và bỏ khóa có số lần 0) — đúng nghĩa **giao multiset**. Ngoài ra `c.elements()` trả về từng phần tử lặp đúng số lần đếm.

In [4]:
from collections import Counter
c1, c2 = Counter("bella"), Counter("label")
print("c1      =", c1)
print("c2      =", c2)
print("c1 & c2 =", c1 & c2)
print("c1 & Counter('roller') =", c1 & Counter("roller"))
print("elements:", list(Counter({'l': 2, 'e': 1}).elements()))

c1      = Counter({'l': 2, 'b': 1, 'e': 1, 'a': 1})
c2      = Counter({'l': 2, 'a': 1, 'b': 1, 'e': 1})
c1 & c2 = Counter({'l': 2, 'b': 1, 'e': 1, 'a': 1})
c1 & Counter('roller') = Counter({'l': 2, 'e': 1})
elements: ['l', 'l', 'e']


### 2.4. Nhân list với số: `["l"] * 2`

`["l"] * 2 == ["l", "l"]`. Dùng để sinh `k` bản sao của một ký tự. Và `res.extend(lst)` nối mọi phần tử của `lst` vào cuối `res`.

In [5]:
res = []
res.extend(["e"] * 1)
res.extend(["l"] * 2)
res.extend(["z"] * 0)       # nhân 0 -> list rỗng, không thêm gì
print(res)

['e', 'l', 'l']


## 3. Cách trực giác (baseline) — duyệt ký tự của từ đầu, "gạch bỏ" ở các từ khác

Với mỗi ký tự `ch` trong từ đầu tiên: nếu **mọi** từ còn lại đều còn chứa `ch`, thì `ch` là ký tự chung — thêm vào kết quả và **xoá một lần** `ch` khỏi mỗi từ khác (để lần lặp sau của `ch` cần thêm một bản nữa).

`s.replace(ch, "", 1)` xoá **lần xuất hiện đầu tiên** của `ch` trong `s` (tham số `1` = tối đa 1 lần).

In [6]:
print("roller".replace("l", "", 1))

def common_chars_brute(words: list[str]) -> list[str]:
    rest = list(words[1:])                    # bản sao để "gạch bỏ"
    res = []
    for ch in words[0]:
        if all(ch in w for w in rest):
            res.append(ch)
            rest = [w.replace(ch, "", 1) for w in rest]
    return res

print(common_chars_brute(["bella", "label", "roller"]))
print(common_chars_brute(["cool", "lock", "cook"]))

roler
['e', 'l', 'l']
['c', 'o']


Độ phức tạp: từ đầu có `L` ký tự; mỗi ký tự kiểm tra và tạo lại `n − 1` chuỗi độ dài `L` → **O(n · L²)**. Với `n, L ≤ 100` vẫn ổn (~10⁶), nhưng tạo chuỗi mới liên tục rất tốn.

## 4. Nút thắt

Ta không cần biết **vị trí** ký tự, chỉ cần biết **mỗi chữ cái xuất hiện bao nhiêu lần** trong mỗi từ. Thay vì sửa chuỗi, hãy **đếm** một lần mỗi từ.

## 5. Insight — số lần chung = min các số lần

Với mỗi chữ cái `c`:

```text
số lần c trong kết quả = min( count_w(c)  với mọi từ w )
```

Giữ một mảng `common[26]`, khởi tạo bằng tần suất của từ đầu tiên, sau đó với mỗi từ tiếp theo: `common[i] = min(common[i], cnt_w[i])`.

In [7]:
words = ["bella", "label", "roller"]
letters = "abelor"
print("     " + "  ".join(letters))
for w in words:
    cnt = count26(w)
    print(f"{w:6} " + "  ".join(str(cnt[ord(ch) - 97]) for ch in letters))
mins = [min(count26(w)[ord(ch) - 97] for w in words) for ch in letters]
print("min    " + "  ".join(str(m) for m in mins))

     a  b  e  l  o  r
bella  1  1  1  2  0  0
label  1  1  1  2  0  0
roller 0  0  1  2  1  2
min    0  0  1  2  0  0


## 6. Lời giải chính — xây từng đoạn rồi ghép

**Đoạn 1:** `common = count26(words[0])`.

**Đoạn 2:** với mỗi từ còn lại, tính `cnt = count26(w)` rồi `common[i] = min(common[i], cnt[i])` cho `i` từ 0 đến 25.

**Đoạn 3:** dựng kết quả: với mỗi `i`, thêm `common[i]` bản sao của chữ cái `chr(i + 97)`.

In [8]:
def common_chars(words: list[str]) -> list[str]:
    common = count26(words[0])                         # Đoạn 1
    for w in words[1:]:                                # Đoạn 2
        cnt = count26(w)
        for i in range(26):
            common[i] = min(common[i], cnt[i])
    res = []                                           # Đoạn 3
    for i in range(26):
        res.extend([chr(i + ord('a'))] * common[i])
    return res

print(common_chars(["bella", "label", "roller"]))
print(common_chars(["cool", "lock", "cook"]))

['e', 'l', 'l']
['c', 'o']


## 7. Trace từng bước

`words = ["cool", "lock", "cook"]` (chỉ hiện các chữ có mặt):

| Sau từ | c | k | l | o | Ghi chú |
|---|:---:|:---:|:---:|:---:|---|
| "cool" (khởi tạo) | 1 | 0 | 1 | 2 | |
| "lock" (cnt: c1 k1 l1 o1) | 1 | 0 | 1 | **1** | o: min(2, 1) |
| "cook" (cnt: c1 k1 o2) | 1 | 0 | **0** | 1 | l: min(1, 0) |

Kết quả: `c × 1`, `o × 1` → `["c", "o"]`.

In [9]:
def trace_common(words: list[str]) -> list[str]:
    show = sorted(set("".join(words)))
    def fmt(arr):
        return "  ".join(f"{ch}={arr[ord(ch) - 97]}" for ch in show)
    common = count26(words[0])
    print(f"khởi tạo từ {words[0]!r:9}: {fmt(common)}")
    for w in words[1:]:
        cnt = count26(w)
        common = [min(a, b_) for a, b_ in zip(common, cnt)]
        print(f"sau {w!r:16}: {fmt(common)}   (cnt của từ: {fmt(cnt)})")
    res = [chr(i + 97) for i in range(26) for _ in range(common[i])]
    print("kết quả:", res)
    return res

trace_common(["cool", "lock", "cook"])
print()
trace_common(["bella", "label", "roller"])

khởi tạo từ 'cool'   : c=1  k=0  l=1  o=2
sau 'lock'          : c=1  k=0  l=1  o=1   (cnt của từ: c=1  k=1  l=1  o=1)
sau 'cook'          : c=1  k=0  l=0  o=1   (cnt của từ: c=1  k=1  l=0  o=2)
kết quả: ['c', 'o']

khởi tạo từ 'bella'  : a=1  b=1  e=1  l=2  o=0  r=0
sau 'label'         : a=1  b=1  e=1  l=2  o=0  r=0   (cnt của từ: a=1  b=1  e=1  l=2  o=0  r=0)
sau 'roller'        : a=0  b=0  e=1  l=2  o=0  r=0   (cnt của từ: a=0  b=0  e=1  l=2  o=1  r=2)
kết quả: ['e', 'l', 'l']


['e', 'l', 'l']

`[min(a, b) for a, b in zip(common, cnt)]` cập nhật cả 26 ô trong một dòng. Và `[chr(i+97) for i in range(26) for _ in range(common[i])]` là comprehension **hai vòng lồng** để sinh mỗi chữ cái `common[i]` lần.

## 8. Code nộp LeetCode

### Cách A — mảng 26 ô lấy min

In [10]:
class Solution:
    def commonChars(self, words: list[str]) -> list[str]:
        common = [float("inf")] * 26
        for w in words:
            cnt = [0] * 26
            for ch in w:
                cnt[ord(ch) - 97] += 1
            for i in range(26):
                common[i] = min(common[i], cnt[i])
        return [chr(i + 97) for i in range(26) for _ in range(common[i])]

print(Solution().commonChars(["bella", "label", "roller"]))
print(Solution().commonChars(["cool", "lock", "cook"]))

['e', 'l', 'l']
['c', 'o']


Mẹo khởi tạo `common = [inf] * 26` (`float("inf")` là **vô cực**, lớn hơn mọi số) giúp xử lý **mọi** từ giống nhau, không cần tách riêng từ đầu. Sau từ đầu tiên, mọi ô đều đã thành số nguyên hữu hạn nên `range(common[i])` an toàn.

### Cách B — Counter và phép giao `&` (ngắn nhất)

In [11]:
class SolutionCounter:
    def commonChars(self, words: list[str]) -> list[str]:
        common = Counter(words[0])
        for w in words[1:]:
            common &= Counter(w)
        return list(common.elements())

print(SolutionCounter().commonChars(["bella", "label", "roller"]))

['e', 'l', 'l']


`common &= Counter(w)` là viết tắt `common = common & Counter(w)`.

Viết gọn hơn nữa bằng `functools.reduce` — hàm "gộp" một dãy bằng một phép toán hai ngôi: `reduce(f, [a, b, c]) = f(f(a, b), c)`.

In [12]:
from functools import reduce
import operator

print(reduce(operator.add, [1, 2, 3, 4]))            # ((1+2)+3)+4
print(reduce(lambda x, y: x * y, [1, 2, 3, 4]))      # ((1*2)*3)*4

class SolutionReduce:
    def commonChars(self, words: list[str]) -> list[str]:
        return list(reduce(operator.and_, map(Counter, words)).elements())

print(SolutionReduce().commonChars(["cool", "lock", "cook"]))

10
24
['c', 'o']


`operator.and_` chính là hàm của toán tử `&` (có dấu `_` vì `and` là từ khóa Python).

### Cách C — với mỗi chữ cái, `min(w.count(ch))`

Chỉ có 26 chữ cái, nên ta có thể hỏi trực tiếp: chữ `ch` xuất hiện ít nhất bao nhiêu lần trong mọi từ? Nên duyệt các chữ cái của **từ đầu tiên** (dùng `set` để khỏi lặp lại).

In [13]:
class SolutionCount:
    def commonChars(self, words: list[str]) -> list[str]:
        res = []
        for ch in sorted(set(words[0])):
            k = min(w.count(ch) for w in words)
            res.extend([ch] * k)
        return res

print(SolutionCount().commonChars(["bella", "label", "roller"]))

['e', 'l', 'l']


## 9. Vì sao đúng?

**Định nghĩa lại bài toán bằng multiset.** Mỗi từ là một multiset chữ cái (chữ cái + số lần). "Ký tự chung kể cả lặp" là **giao** của các multiset: chữ `c` thuộc giao `k` lần ⇔ mọi từ có ít nhất `k` chữ `c` ⇔ `k ≤ min_w count_w(c)`. Lấy `k` lớn nhất có thể → `k = min_w count_w(c)`.

**Invariant (Cách A).** Sau khi xử lý `t` từ đầu, `common[i] = min(count_{w_1}(i), ..., count_{w_t}(i))`. Ban đầu (`inf`) là min của tập rỗng; mỗi bước `min(common[i], cnt[i])` thêm một từ vào. Sau `n` từ, `common` đúng là multiset giao.

**Cách brute force.** Mỗi khi chọn `ch`, ta xoá một bản `ch` ở các từ khác — tương đương "dùng hết" một đơn vị đếm; `ch` được chọn tối đa `min` lần.

## 10. Độ phức tạp

Gọi `n` = số từ, `L` = độ dài tối đa, `S = n·L` = tổng số ký tự, `Σ = 26`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| Brute force xoá dần | O(n·L²) | O(S) | tạo chuỗi mới liên tục |
| **A. Mảng 26 ô** | O(S + n·Σ) | O(Σ) | chuẩn, không phụ thuộc thư viện |
| B. Counter `&` | O(S + n·Σ) | O(Σ) | ngắn nhất |
| C. `min(count)` | O(Σ · S) | O(Σ) | mỗi chữ cái quét lại mọi từ; vẫn nhanh vì Σ = 26 |

(Kích thước kết quả không vượt quá `L`.)

## 11. Bộ test

Thứ tự kết quả tuỳ ý nên khi so sánh ta **sắp xếp** cả hai phía.

In [14]:
tests = [
    (["bella", "label", "roller"], ["e", "l", "l"]),
    (["cool", "lock", "cook"], ["c", "o"]),
    (["abc"], ["a", "b", "c"]),                     # một từ -> chính nó
    (["aaa", "aa", "a"], ["a"]),
    (["abc", "def"], []),                           # không chung gì
    (["zz", "zzz", "zzzz"], ["z", "z"]),
    (["acabcddd", "bcbdbcbd", "baddbadb", "cbdddcac", "aacbcccd", "ccccddda", "cababaab", "addcaccd"], []),
]
impls = {
    "brute": common_chars_brute, "chính": common_chars,
    "LC-A": Solution().commonChars, "LC-B": SolutionCounter().commonChars,
    "LC-reduce": SolutionReduce().commonChars, "LC-C": SolutionCount().commonChars,
}
for words, expected in tests:
    for name, f in impls.items():
        got = f(words)
        assert sorted(got) == sorted(expected), (name, words, got, expected)
    print(f"{str(words)[:40]:42} -> {expected} PASS")
print("Tất cả test PASS!")

['bella', 'label', 'roller']               -> ['e', 'l', 'l'] PASS
['cool', 'lock', 'cook']                   -> ['c', 'o'] PASS
['abc']                                    -> ['a', 'b', 'c'] PASS
['aaa', 'aa', 'a']                         -> ['a'] PASS
['abc', 'def']                             -> [] PASS
['zz', 'zzz', 'zzzz']                      -> ['z', 'z'] PASS
['acabcddd', 'bcbdbcbd', 'baddbadb', 'cb   -> [] PASS
Tất cả test PASS!


In [15]:
import random
rng = random.Random(1002)
for _ in range(2000):
    words = ["".join(rng.choice("abcd") for _ in range(rng.randint(1, 8))) for _ in range(rng.randint(1, 5))]
    expected = sorted(common_chars_brute(words))
    for name, f in impls.items():
        assert sorted(f(words)) == expected, (name, words)
print("2000 test ngẫu nhiên: mọi cách khớp")

2000 test ngẫu nhiên: mọi cách khớp


## 12. Lỗi người mới hay gặp

### Lỗi 1: Dùng `set` → mất thông tin lặp

In [16]:
def wrong_set(words):
    common = set(words[0])
    for w in words[1:]:
        common &= set(w)
    return sorted(common)

print("Sai :", wrong_set(["bella", "label", "roller"]))
print("Đúng:", sorted(common_chars(["bella", "label", "roller"])))

Sai : ['e', 'l']
Đúng: ['e', 'l', 'l']


**Vì sao sai:** `set` chỉ nhớ "có / không", bỏ mất số lần → chỉ ra một chữ `l`. **Sửa:** dùng Counter hoặc mảng đếm (multiset).

### Lỗi 2: Lấy `max` thay vì `min`, hoặc cộng dồn

In [17]:
def wrong_max(words):
    common = count26(words[0])
    for w in words[1:]:
        common = [max(a, b_) for a, b_ in zip(common, count26(w))]
    return [chr(i + 97) for i in range(26) for _ in range(common[i])]

print("Sai :", wrong_max(["cool", "lock", "cook"]))

Sai : ['c', 'k', 'l', 'o', 'o']


**Vì sao sai:** `max` là phép **hợp** multiset (ký tự có trong *ít nhất một* từ). Giao cần `min`.

### Lỗi 3: Kiểm tra `ch in w` cho mỗi ký tự của từ đầu mà không "gạch bỏ"

In [18]:
def wrong_no_remove(words):
    return [ch for ch in words[0] if all(ch in w for w in words[1:])]

print("Sai :", wrong_no_remove(["cool", "lock", "cook"]))
print("Đúng:", common_chars(["cool", "lock", "cook"]))

Sai : ['c', 'o', 'o']
Đúng: ['c', 'o']


**Vì sao sai:** chữ `o` thứ hai của "cool" vẫn thấy `o` trong "lock" (vì chưa xoá bản đã dùng) → đếm dư. **Sửa:** xoá một bản sau mỗi lần dùng (`replace(ch, "", 1)`) hoặc đếm tần suất.

### Lỗi 4: Khởi tạo `common = [0] * 26`

`min(0, x) = 0` với mọi `x` → kết quả luôn rỗng. Phải khởi tạo bằng **tần suất của từ đầu** hoặc **vô cực**.

### Lỗi 5: Dùng `Counter` rồi trả `list(common)`

`list(Counter)` chỉ cho các **khóa** (mỗi chữ một lần). Phải dùng `common.elements()`.

In [19]:
cc = Counter("bella") & Counter("label") & Counter("roller")
print("list(cc)          =", list(cc))
print("list(cc.elements())=", list(cc.elements()))

list(cc)          = ['e', 'l']
list(cc.elements())= ['e', 'l', 'l']


## 13. Mẹo nhận dạng pattern

- "Ký tự/phần tử **chung** của nhiều dãy, **tính cả lặp**" → **giao multiset** = `min` tần suất (Counter `&`).
- "Có trong **ít nhất một**, tính cả lặp" → hợp multiset = `max` tần suất (Counter `|`).
- Chỉ chữ cái thường → **mảng 26 ô** với `ord(ch) - ord('a')` nhanh và gọn.
- Kết quả "thứ tự tuỳ ý" → khi test hãy so `sorted(...)` hoặc so Counter.

## 14. Biến thể

1. **LeetCode 349 / 350 — Intersection of Two Arrays (I/II):** giao **set** (không lặp) và giao **multiset** (có lặp) của hai mảng số.
2. **LeetCode 383 — Ransom Note:** kiểm tra multiset này có **chứa** multiset kia (`Counter(a) <= Counter(b)` từ Python 3.10).
3. **LeetCode 242 — Valid Anagram:** hai multiset **bằng nhau**.
4. **LeetCode 1160 — Find Words That Can Be Formed by Characters.**

In [20]:
print("Giao multiset [1,2,2,1] & [2,2]:", list((Counter([1, 2, 2, 1]) & Counter([2, 2])).elements()))
print("Hợp  multiset 'aab' | 'abb':", sorted((Counter("aab") | Counter("abb")).elements()))
print("Ransom note 'aa' từ 'aab'?", not (Counter("aa") - Counter("aab")))

Giao multiset [1,2,2,1] & [2,2]: [2, 2]
Hợp  multiset 'aab' | 'abb': ['a', 'a', 'b', 'b']
Ransom note 'aa' từ 'aab'? True


`Counter(a) - Counter(b)` giữ lại phần **dư** của `a` (chỉ số dương). Nếu rỗng (falsy) thì `a` nằm gọn trong `b`.

## 15. Bài tập tự luyện (TODO)

### Bài 1 — Đếm 26 ô
Viết `freq26(word)` trả về list 26 số đếm (không dùng Counter).

In [21]:
def freq26(word: str) -> list[int]:
    # TODO
    pass

# assert freq26("abca")[:3] == [2, 1, 1]
# assert sum(freq26("hello")) == 5

### Bài 2 — Giao hai mảng có lặp (LeetCode 350)

In [22]:
def intersect(nums1: list[int], nums2: list[int]) -> list[int]:
    # TODO
    pass

# assert sorted(intersect([1, 2, 2, 1], [2, 2])) == [2, 2]
# assert sorted(intersect([4, 9, 5], [9, 4, 9, 8, 4])) == [4, 9]

### Bài 3 — Ký tự có trong ít nhất một từ (hợp multiset)
Trả về list (đã sắp xếp) các ký tự, mỗi ký tự lặp `max` số lần qua các từ.

In [23]:
def union_chars(words: list[str]) -> list[str]:
    # TODO
    pass

# assert union_chars(["aab", "abb"]) == ["a", "a", "b", "b"]

### Bài 4 — Ransom Note (LeetCode 383)
`can_construct(note, magazine)`: mỗi chữ trong `magazine` chỉ dùng được một lần.

In [24]:
def can_construct(note: str, magazine: str) -> bool:
    # TODO
    pass

# assert can_construct("aa", "aab") == True
# assert can_construct("aa", "ab") == False

### Bài 5 — Ký tự chung với số lần tối thiểu
Trả về dict `{ký_tự: số_lần}` của các ký tự xuất hiện **ít nhất `k` lần trong mỗi từ** (giá trị là số lần chung `min`).

In [25]:
def common_at_least(words: list[str], k: int) -> dict:
    # TODO
    pass

# assert common_at_least(["bella", "label", "roller"], 2) == {'l': 2}
# assert common_at_least(["bella", "label", "roller"], 1) == {'e': 1, 'l': 2}

### Bài 6 (nâng cao) — LeetCode 1160
Cho `words` và `chars`, trả về **tổng độ dài** các từ ghép được từ `chars` (mỗi ký tự trong `chars` dùng một lần cho mỗi từ).

In [26]:
def count_characters(words: list[str], chars: str) -> int:
    # TODO
    pass

# assert count_characters(["cat", "bt", "hat", "tree"], "atach") == 6
# assert count_characters(["hello", "world", "leetcode"], "welldonehoneyr") == 10

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | `[0] * 26` | chỉ số `ord(ch) - 97` | `cnt[...] += 1` |
| 2 | giao multiset | `Counter(nums1) & Counter(nums2)` | `list(....elements())` |
| 3 | hợp multiset | toán tử `|` | `sorted(reduce(operator.or_, map(Counter, words)).elements())` |
| 4 | note ⊆ magazine | so từng chữ: `cnt_note[c] <= cnt_mag[c]` | `all(...)` trên các khóa của note |
| 5 | tính giao như bài chính | lọc `v >= k` | `{c: v for c, v in common.items() if v >= k}` |
| 6 | mỗi từ là một ransom note | `Counter(chars)` một lần | cộng `len(w)` nếu ghép được |

## 17. Đáp án tham khảo

In [27]:
def ans_freq26(word):
    cnt = [0] * 26
    for ch in word:
        cnt[ord(ch) - 97] += 1
    return cnt

def ans_intersect(nums1, nums2):
    return list((Counter(nums1) & Counter(nums2)).elements())

def ans_union_chars(words):
    return sorted(reduce(operator.or_, map(Counter, words)).elements())

def ans_can_construct(note, magazine):
    cm = Counter(magazine)
    return all(cm[c] >= v for c, v in Counter(note).items())

def ans_common_at_least(words, k):
    common = reduce(operator.and_, map(Counter, words))
    return {c: v for c, v in sorted(common.items()) if v >= k}

def ans_count_characters(words, chars):
    pool = Counter(chars)                       # đếm chars một lần
    return sum(len(w) for w in words if all(pool[c] >= v for c, v in Counter(w).items()))

assert ans_freq26("abca")[:3] == [2, 1, 1]
assert sum(ans_freq26("hello")) == 5
assert sorted(ans_intersect([1, 2, 2, 1], [2, 2])) == [2, 2]
assert sorted(ans_intersect([4, 9, 5], [9, 4, 9, 8, 4])) == [4, 9]
assert ans_union_chars(["aab", "abb"]) == ["a", "a", "b", "b"]
assert ans_can_construct("aa", "aab") == True
assert ans_can_construct("aa", "ab") == False
assert ans_common_at_least(["bella", "label", "roller"], 2) == {'l': 2}
assert ans_common_at_least(["bella", "label", "roller"], 1) == {'e': 1, 'l': 2}
assert ans_count_characters(["cat", "bt", "hat", "tree"], "atach") == 6
assert ans_count_characters(["hello", "world", "leetcode"], "welldonehoneyr") == 10
print("Đáp án: mọi test PASS!")

Đáp án: mọi test PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def commonChars(self, words: list[str]) -> list[str]:
        common = Counter(words[0])
        for w in words[1:]:
            common &= Counter(w)          # giao multiset = min tần suất
        return list(common.elements())
```

Bản không dùng thư viện: mảng `common[26]` khởi tạo `inf`, mỗi từ cập nhật `min`, cuối cùng sinh `chr(i+97)` đúng `common[i]` lần.

**Checklist trước khi submit**
- [ ] Giữ **số lần lặp** (không dùng `set`)?
- [ ] Dùng `min` (giao), không phải `max` (hợp)?
- [ ] Khởi tạo bằng từ đầu tiên hoặc vô cực, không phải 0?
- [ ] Với Counter: trả `elements()`, không phải `list(counter)`?
- [ ] Một từ duy nhất → trả mọi ký tự của từ đó?